# 01 – Build the dataset

Downloads everything phase 1 needs and stores it in `data/processed/`:

1. **Universe** – current S&P 500 members → top N by market cap (`config.yaml`)
2. **Stock splits** – to restate per-share values (EPS) to today's share basis
3. **Annual fundamentals** – from SEC XBRL `companyfacts`, 10-K only, *as first reported*
4. **Data quality check** – which metrics are missing for which company

All HTTP responses are cached in `data/raw/`, so re-running is fast and does not hit the SEC again
(cache expires after `cache.max_age_days`).

In [ ]:
from IPython.display import display
import pandas as pd

from stockpred.config import load_settings
from stockpred.edgar import EdgarClient
from stockpred.universe import fetch_sp500, add_market_cap, top_n
from stockpred.prices import split_history
from stockpred.fundamentals import build_panel

pd.set_option("display.float_format", "{:,.2f}".format)
settings = load_settings()
client = EdgarClient(
    user_agent=settings.sec_user_agent,
    cache_dir=settings.raw_dir / "edgar",
    max_requests_per_second=settings.max_requests_per_second,
    max_age_days=settings.max_age_days,
)
settings

## 1. Universe

Wikipedia's S&P 500 table already contains each company's **CIK** (the SEC company ID), so no fuzzy name matching is needed.
Market cap = latest shares outstanding (one SEC *frames* request for all companies) × last close (one batch `yfinance` request).
Multi-class companies (GOOGL, META, BRK.B, V…) are missing in SEC frames → fallback to Yahoo market cap.

⚠️ This is **today's** index membership → survivorship bias. Historical constituents come in phase 4.

In [ ]:
sp500 = fetch_sp500(settings.sec_user_agent, settings.raw_dir / "sp500_constituents.csv")
sp500 = add_market_cap(sp500, client)
universe = top_n(sp500, settings.top_n)
universe.to_csv(settings.processed_dir / "universe.csv", index=False)

print(f"S&P 500 rows: {len(sp500)}, missing market cap: {sp500['market_cap'].isna().sum()}")
display(universe[["ticker", "name", "sector", "market_cap", "market_cap_source"]].head(15))

## 2. Stock splits

10-K filings report EPS on the share basis valid *at filing time*. Apple's FY2012 EPS was $44.15 before the 7:1 and 4:1 splits.
We divide each year's EPS by all split ratios that happened after that year's filing date.

In [ ]:
splits = split_history(universe["ticker"].tolist(), settings.raw_dir / "splits.parquet")
print(f"Companies with at least one split: {len(splits)}")

## 3. Annual fundamentals

For every company: one SEC request (cached), then `company_fundamentals()` picks annual 10-K values,
handles different XBRL tags for the same concept and adds derived metrics (margins, FCF, ROE, growth).
`available_date` = filing date of the original 10-K → used later to avoid look-ahead bias in backtests.

Companies that moved to a new legal entity (e.g. ExxonMobil Holdings Corp, 2026) keep their history under
the old CIK → `cik_predecessors` in `config.yaml` merges both. A warning lists any company with no data.

In [ ]:
panel = build_panel(client, universe, start_year=settings.start_year, splits=splits,
                    predecessors=settings.cik_predecessors)
panel.to_parquet(settings.processed_dir / "fundamentals_annual.parquet", index=False)
print(panel.shape)
display(panel[panel["ticker"] == universe.loc[0, "ticker"]].tail(5))

## 4. Data quality check

Share of fiscal years (since `start_year`) with a value. Expect gaps, e.g.:
* **banks / insurers** have no gross profit and often a different revenue concept,
* **R&D** is not reported by many non-tech companies,
* companies listed recently have short histories.

Low coverage is not a bug in itself – but check it before trusting a chart.

In [ ]:
key_metrics = ["revenue", "net_income", "operating_income", "gross_profit",
               "operating_cash_flow", "capex", "eps_diluted", "equity"]
coverage = (panel.groupby("ticker")[key_metrics]
                 .agg(lambda s: s.notna().mean())
                 .sort_values("revenue"))
display(coverage.style.format("{:.0%}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))